In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Carregamento inicial de dados

import pandas as pd
import numpy as np

ANO_MINIMO = 2015

paths = {
    'ind_10_19': '/content/drive/MyDrive/API-6/indicadores-continuidade-coletivos-2010-2019.csv',
    'ind_20_29': '/content/drive/MyDrive/API-6/indicadores-continuidade-coletivos-2020-2029.csv',
    'atributos': '/content/drive/MyDrive/API-6/indicadores-continuidade-coletivos-atributos.csv',
    'limites':   '/content/drive/MyDrive/API-6/indicadores-continuidade-coletivos-limite.csv'
}

def load_csv(path):
    try:
        return pd.read_csv(path, encoding='latin1', sep=';', low_memory=False)
    except Exception as e:
        print(f'Erro ao carregar {path}: {e}')
        return pd.DataFrame()

df_10_19 = load_csv(paths['ind_10_19'])
df_20_29 = load_csv(paths['ind_20_29'])
df_atrib = load_csv(paths['atributos'])
df_limit = load_csv(paths['limites'])

print('Arquivos carregados com sucesso.')
print(f'df_10_19: {df_10_19.shape}')
print(f'df_20_29: {df_20_29.shape}')

Arquivos carregados com sucesso.
df_10_19: (8554625, 9)
df_20_29: (4786829, 9)


In [3]:
# 2. Limpeza e consolidação dos indicadores

def clean_df(df, val_col, date_col='DatGeracaoConjuntoDados'):
    if df.empty:
        return df
    df = df.copy()
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    if val_col in df.columns:
        df[val_col] = df[val_col].astype(str).str.replace(',', '.', regex=False)
        df[val_col] = pd.to_numeric(df[val_col], errors='coerce')
    cols_fix = ['NumCNPJ', 'IdeConjUndConsumidoras', 'AnoIndice', 'NumPeriodoIndice', 'AnoLimiteQualidade']
    for col in cols_fix:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'SigAgente' in df.columns:
        df['SigAgente'] = df['SigAgente'].astype(str).str.strip().str.upper()
    return df

df_10_19 = clean_df(df_10_19, 'VlrIndiceEnviado')
df_20_29 = clean_df(df_20_29, 'VlrIndiceEnviado')
df_atrib = clean_df(df_atrib, 'VlrIndiceEnviado')

df_10_29 = pd.concat([df_10_19, df_20_29], ignore_index=True)

df_limit_raw = load_csv(paths['limites'])
df_limit_processed = clean_df(df_limit_raw.copy(), 'VlrLimite')

if 'AnoIndice' in df_10_29.columns:
    df_10_29 = df_10_29[df_10_29['AnoIndice'] >= ANO_MINIMO].copy()
if 'AnoIndice' in df_atrib.columns:
    df_atrib = df_atrib[df_atrib['AnoIndice'] >= ANO_MINIMO].copy()
if 'AnoLimiteQualidade' in df_limit_processed.columns:
    df_limit_processed = df_limit_processed[df_limit_processed['AnoLimiteQualidade'] >= ANO_MINIMO].copy()

df_limit_processed = df_limit_processed.rename(
    columns={'AnoLimiteQualidade': 'AnoIndice', 'VlrLimite': 'VlrIndiceEnviado'}
)
df_limit_processed['TipoRegistro'] = 'Limite'

df_limit_dec = df_limit_processed.copy(); df_limit_dec['SigIndicador'] = 'DEC'
df_limit_fec = df_limit_processed.copy(); df_limit_fec['SigIndicador'] = 'FEC'
df_limit = pd.concat([df_limit_dec, df_limit_fec], ignore_index=True)

df_apurados = pd.concat([df_10_29, df_atrib], ignore_index=True)
df_apurados['TipoRegistro'] = 'Apurado'

common_cols = [
    'IdeConjUndConsumidoras', 'SigIndicador', 'AnoIndice', 'NumPeriodoIndice',
    'VlrIndiceEnviado', 'TipoRegistro'
]

df_indicadores = pd.concat([
    df_apurados[[c for c in common_cols if c in df_apurados.columns]],
    df_limit[[c for c in common_cols if c in df_limit.columns]]
], ignore_index=True)

df_indicadores['IdeConjUndConsumidoras'] = df_indicadores['IdeConjUndConsumidoras'].astype(int)
df_indicadores['AnoIndice'] = df_indicadores['AnoIndice'].astype(int)
df_indicadores['NumPeriodoIndice_key'] = pd.to_numeric(
    df_indicadores['NumPeriodoIndice'], errors='coerce'
).fillna(0).astype(int)

In [ ]:
# 3. Instalação do Prophet
# Prophet é muito mais leve que redes neurais: sem GPU, sem batches,
# fit em segundos por série — ideal para 6000+ unidades.

!pip install prophet -q

In [ ]:
# 4. Imports para Prophet

import os
import gc
import logging
import warnings
from prophet import Prophet

warnings.filterwarnings('ignore')
# Silencia logs verbosos do Prophet/Stan
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

## INÍCIO DE TRATAMENTO PARA SÉRIES TEMPORAIS E TREINAMENTO COM PROPHET - DEC

In [ ]:
# 5. Preparação da série DEC

df_dec = df_indicadores[df_indicadores['SigIndicador'] == 'DEC'].copy()
df_dec = df_dec.dropna(subset=['NumPeriodoIndice'])
df_dec['data'] = pd.to_datetime(
    df_dec['AnoIndice'].astype(int).astype(str) + '-' +
    df_dec['NumPeriodoIndice_key'].astype(int).astype(str).str.zfill(2) + '-01'
)

data_max = '2026-02-01'
ids_validos = df_dec.loc[df_dec['data'] == data_max, 'IdeConjUndConsumidoras'].unique()

dec_por_agente = (
    df_dec[df_dec['IdeConjUndConsumidoras'].isin(ids_validos)]
    .sort_values(['IdeConjUndConsumidoras', 'data'])
)

print(f'Unidades DEC válidas: {len(ids_validos)}')
dec_por_agente.head()

Unidades DEC válidas: 2740


,IdeConjUndConsumidoras,SigIndicador,AnoIndice,NumPeriodoIndice,VlrIndiceEnviado,TipoRegistro,NumPeriodoIndice_key,data
627874,964,DEC,2015,1.0,0.49,Apurado,1,2015-01-01
636415,964,DEC,2015,2.0,1.17,Apurado,2,2015-02-01
634006,964,DEC,2015,3.0,1.20,Apurado,3,2015-03-01
615391,964,DEC,2015,4.0,0.54,Apurado,4,2015-04-01
607507,964,DEC,2015,5.0,1.25,Apurado,5,2015-05-01


In [ ]:
# 6. Parâmetros e função Prophet para DEC
#
# Configuração mínima e eficiente:
#   - yearly_seasonality=True  → captura padrão anual (DEC/FEC têm forte sazonalidade)
#   - weekly_seasonality=False → dados mensais, não faz sentido
#   - daily_seasonality=False  → idem
#   - n_changepoints=10        → reduzido para evitar overfitting em séries curtas
#   - Sem regressores extras   → mantém velocidade para 6 000+ unidades

HORIZONTE   = 12   # meses a prever
MIN_AMOSTRAS = 24  # mínimo de pontos históricos (2 anos)

def processar_unidade_prophet(cod, df_unit, indicador='DEC'):
    df_unit = (
        df_unit[['data', 'VlrIndiceEnviado']]
        .dropna()
        .rename(columns={'data': 'ds', 'VlrIndiceEnviado': 'y'})
        .reset_index(drop=True)
    )

    if len(df_unit) < MIN_AMOSTRAS:
        return None  # histórico insuficiente

    # Treina com todos os dados disponíveis
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        n_changepoints=10,
        seasonality_mode='additive',
    )
    m.fit(df_unit)

    # Calcula MAE no último HORIZONTE meses (validação interna)
    if len(df_unit) >= HORIZONTE + MIN_AMOSTRAS:
        df_train_val = df_unit.iloc[:-HORIZONTE]
        df_test_val  = df_unit.iloc[-HORIZONTE:]
        m_val = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            n_changepoints=10,
            seasonality_mode='additive',
        )
        m_val.fit(df_train_val)
        future_val = m_val.make_future_dataframe(periods=HORIZONTE, freq='MS', include_history=False)
        forecast_val = m_val.predict(future_val)
        mae_val = float(np.mean(np.abs(forecast_val['yhat'].values - df_test_val['y'].values)))
    else:
        mae_val = float('nan')

    # Previsão futura com o modelo completo
    future = m.make_future_dataframe(periods=HORIZONTE, freq='MS', include_history=False)
    forecast = m.predict(future)

    df_prev = pd.DataFrame({
        'cod_unidade': cod,
        'indicador': indicador,
        'mae': round(mae_val, 6),
        'data': forecast['ds'],
        'previsao': forecast['yhat'].round(6),
        'previsao_lower': forecast['yhat_lower'].round(6),
        'previsao_upper': forecast['yhat_upper'].round(6),
    })

    return {'cod_unidade': cod, 'mae': mae_val, 'previsoes': df_prev}

In [ ]:
# 7. Treinamento Prophet para todas as unidades - DEC
#
# Salva parcialmente a cada CHECKPOINT unidades para não perder
# progresso em caso de queda do runtime (muito comum com 6 000+ treinos).

CAMINHO_PREVISOES_DEC = '/content/drive/MyDrive/API-6/previsoes_dec_prophet_todas_unidades.csv'
CHECKPOINT = 500  # salva CSV parcial a cada N unidades

todas_unidades_dec   = dec_por_agente['IdeConjUndConsumidoras'].unique()
resultados_dec       = []
todas_previsoes_dec  = []

for i, cod in enumerate(todas_unidades_dec, start=1):
    df_unit = dec_por_agente[dec_por_agente['IdeConjUndConsumidoras'] == cod].copy()
    try:
        res = processar_unidade_prophet(cod, df_unit, indicador='DEC')
        if res:
            resultados_dec.append({'cod_unidade': res['cod_unidade'], 'mae': res['mae']})
            todas_previsoes_dec.append(res['previsoes'])
            print(f"✓ [{i}/{len(todas_unidades_dec)}] Unidade {cod} | MAE: {res['mae']:.4f}")
        else:
            print(f"⚠ [{i}/{len(todas_unidades_dec)}] Unidade {cod} | ignorada (dados insuficientes)")
    except Exception as e:
        print(f"✗ [{i}/{len(todas_unidades_dec)}] Unidade {cod} | erro: {e}")
    finally:
        gc.collect()

    # Checkpoint: salva CSV parcial periodicamente
    if i % CHECKPOINT == 0 and todas_previsoes_dec:
        pd.concat(todas_previsoes_dec, ignore_index=True).assign(
            data=lambda d: d['data'].dt.strftime('%Y-%m-%d')
        ).to_csv(CAMINHO_PREVISOES_DEC, sep=';', decimal=',', index=False)
        print(f"  → Checkpoint salvo ({i} unidades processadas)")

✓ [1/2740] Unidade 964 | MAE: 0.2643
✓ [2/2740] Unidade 2142 | MAE: 0.1766
✓ [3/2740] Unidade 2922 | MAE: 0.1279
✓ [4/2740] Unidade 5273 | MAE: 0.2447
✓ [5/2740] Unidade 12474 | MAE: 0.3276
✓ [6/2740] Unidade 12475 | MAE: 1.0682
✓ [7/2740] Unidade 12476 | MAE: 0.0864
✓ [8/2740] Unidade 12477 | MAE: 0.6200
✓ [9/2740] Unidade 12478 | MAE: 0.7503
✓ [10/2740] Unidade 12479 | MAE: 0.2749
✓ [11/2740] Unidade 12481 | MAE: 0.8753
✓ [12/2740] Unidade 12482 | MAE: 0.2086
✓ [13/2740] Unidade 12483 | MAE: 0.4704
✓ [14/2740] Unidade 12484 | MAE: 0.6533
✓ [15/2740] Unidade 12485 | MAE: 0.5246
✓ [16/2740] Unidade 12486 | MAE: 2.0182
✓ [17/2740] Unidade 12487 | MAE: 1.0992
✓ [18/2740] Unidade 12488 | MAE: 0.9380
✓ [19/2740] Unidade 12489 | MAE: 1.8931
✓ [20/2740] Unidade 12524 | MAE: 2.1562
✓ [21/2740] Unidade 12525 | MAE: 1.6937
✓ [22/2740] Unidade 12532 | MAE: 1.1469
✓ [23/2740] Unidade 12539 | MAE: 1.1747
✓ [24/2740] Unidade 12540 | MAE: 0.5533
✓ [25/2740] Unidade 12541 | MAE: 2.3173
✓ [26/2740] Un

In [ ]:
# 8. Exportação final de resultados - DEC

if todas_previsoes_dec:
    df_todas_previsoes_dec = pd.concat(todas_previsoes_dec, ignore_index=True)
    df_todas_previsoes_dec['data'] = df_todas_previsoes_dec['data'].dt.strftime('%Y-%m-%d')
    df_todas_previsoes_dec.to_csv(
        CAMINHO_PREVISOES_DEC,
        sep=';', decimal=',', index=False
    )
    print(f'✓ CSV final salvo em: {CAMINHO_PREVISOES_DEC}')
    print(f'  Total de linhas     : {len(df_todas_previsoes_dec)}')
    print(f'  Unidades processadas: {df_todas_previsoes_dec["cod_unidade"].nunique()}')
    print()
    print(df_todas_previsoes_dec.head(10))
else:
    print('⚠ Nenhuma previsão DEC gerada para exportar.')

✓ CSV final salvo em: /content/drive/MyDrive/API-6/previsoes_dec_prophet_todas_unidades.csv
  Total de linhas     : 30768
  Unidades processadas: 2564

   cod_unidade indicador       mae        data  previsao  previsao_lower  \
0          964       DEC  0.264269  2026-03-01  0.667080        0.316575   
1          964       DEC  0.264269  2026-04-01  0.255178       -0.065189   
2          964       DEC  0.264269  2026-05-01  0.335213       -0.014525   
3          964       DEC  0.264269  2026-06-01  0.213436       -0.131305   
4          964       DEC  0.264269  2026-07-01  0.265024       -0.115456   
5          964       DEC  0.264269  2026-08-01  0.224720       -0.103880   
6          964       DEC  0.264269  2026-09-01  0.305017       -0.035240   
7          964       DEC  0.264269  2026-10-01  0.582870        0.238661   
8          964       DEC  0.264269  2026-11-01  0.476119        0.090450   
9          964       DEC  0.264269  2026-12-01  0.674364        0.307720   

   previsao

## INÍCIO DE TRATAMENTO PARA SÉRIES TEMPORAIS E TREINAMENTO COM PROPHET - FEC

In [ ]:
# 9. Preparação da série FEC

df_fec = df_indicadores[df_indicadores['SigIndicador'] == 'FEC'].copy()
df_fec = df_fec.dropna(subset=['NumPeriodoIndice'])
df_fec['data'] = pd.to_datetime(
    df_fec['AnoIndice'].astype(int).astype(str) + '-' +
    df_fec['NumPeriodoIndice_key'].astype(int).astype(str).str.zfill(2) + '-01'
)

data_max_fec = '2026-02-01'
ids_validos_fec = df_fec.loc[df_fec['data'] == data_max_fec, 'IdeConjUndConsumidoras'].unique()

fec_por_agente = (
    df_fec[df_fec['IdeConjUndConsumidoras'].isin(ids_validos_fec)]
    .sort_values(['IdeConjUndConsumidoras', 'data'])
)

print(f'Unidades FEC válidas: {len(ids_validos_fec)}')
fec_por_agente.head()

Unidades FEC válidas: 2740


,IdeConjUndConsumidoras,SigIndicador,AnoIndice,NumPeriodoIndice,VlrIndiceEnviado,TipoRegistro,NumPeriodoIndice_key,data
610135,964,FEC,2015,1.0,0.50,Apurado,1,2015-01-01
614515,964,FEC,2015,2.0,1.18,Apurado,2,2015-02-01
621085,964,FEC,2015,3.0,0.84,Apurado,3,2015-03-01
612763,964,FEC,2015,4.0,0.13,Apurado,4,2015-04-01
636196,964,FEC,2015,5.0,0.92,Apurado,5,2015-05-01


In [ ]:
# 10. Treinamento Prophet para todas as unidades - FEC
#
# Reutiliza a mesma função processar_unidade_prophet — apenas passa indicador='FEC'

CAMINHO_PREVISOES_FEC = '/content/drive/MyDrive/API-6/previsoes_fec_prophet_todas_unidades.csv'

todas_unidades_fec  = fec_por_agente['IdeConjUndConsumidoras'].unique()
resultados_fec      = []
todas_previsoes_fec = []

for i, cod in enumerate(todas_unidades_fec, start=1):
    df_unit = fec_por_agente[fec_por_agente['IdeConjUndConsumidoras'] == cod].copy()
    try:
        res = processar_unidade_prophet(cod, df_unit, indicador='FEC')
        if res:
            resultados_fec.append({'cod_unidade': res['cod_unidade'], 'mae': res['mae']})
            todas_previsoes_fec.append(res['previsoes'])
            print(f"✓ [{i}/{len(todas_unidades_fec)}] Unidade {cod} | MAE: {res['mae']:.4f}")
        else:
            print(f"⚠ [{i}/{len(todas_unidades_fec)}] Unidade {cod} | ignorada (dados insuficientes)")
    except Exception as e:
        print(f"✗ [{i}/{len(todas_unidades_fec)}] Unidade {cod} | erro: {e}")
    finally:
        gc.collect()

    # Checkpoint parcial
    if i % CHECKPOINT == 0 and todas_previsoes_fec:
        pd.concat(todas_previsoes_fec, ignore_index=True).assign(
            data=lambda d: d['data'].dt.strftime('%Y-%m-%d')
        ).to_csv(CAMINHO_PREVISOES_FEC, sep=';', decimal=',', index=False)
        print(f"  → Checkpoint salvo ({i} unidades processadas)")

✓ [1/2740] Unidade 964 | MAE: 0.2003
✓ [2/2740] Unidade 2142 | MAE: 0.2284
✓ [3/2740] Unidade 2922 | MAE: 0.1007
✓ [4/2740] Unidade 5273 | MAE: 0.1670
✓ [5/2740] Unidade 12474 | MAE: 0.3429
✓ [6/2740] Unidade 12475 | MAE: 0.5598
✓ [7/2740] Unidade 12476 | MAE: 0.0967
✓ [8/2740] Unidade 12477 | MAE: 0.3403
✓ [9/2740] Unidade 12478 | MAE: 0.4319
✓ [10/2740] Unidade 12479 | MAE: 0.3602
✓ [11/2740] Unidade 12481 | MAE: 0.5462
✓ [12/2740] Unidade 12482 | MAE: 0.2100
✓ [13/2740] Unidade 12483 | MAE: 0.2554
✓ [14/2740] Unidade 12484 | MAE: 0.4827
✓ [15/2740] Unidade 12485 | MAE: 0.2992
✓ [16/2740] Unidade 12486 | MAE: 0.6310
✓ [17/2740] Unidade 12487 | MAE: 0.6128
✓ [18/2740] Unidade 12488 | MAE: 0.5004
✓ [19/2740] Unidade 12489 | MAE: 1.2576
✓ [20/2740] Unidade 12524 | MAE: 0.4100
✓ [21/2740] Unidade 12525 | MAE: 0.5228
✓ [22/2740] Unidade 12532 | MAE: 0.5248
✓ [23/2740] Unidade 12539 | MAE: 0.1759
✓ [24/2740] Unidade 12540 | MAE: 0.2919
✓ [25/2740] Unidade 12541 | MAE: 0.5916
✓ [26/2740] Un

In [ ]:
# 11. Exportação final de resultados - FEC

if todas_previsoes_fec:
    df_todas_previsoes_fec = pd.concat(todas_previsoes_fec, ignore_index=True)
    df_todas_previsoes_fec['data'] = df_todas_previsoes_fec['data'].dt.strftime('%Y-%m-%d')
    df_todas_previsoes_fec.to_csv(
        CAMINHO_PREVISOES_FEC,
        sep=';', decimal=',', index=False
    )
    print(f'✓ CSV final salvo em: {CAMINHO_PREVISOES_FEC}')
    print(f'  Total de linhas     : {len(df_todas_previsoes_fec)}')
    print(f'  Unidades processadas: {df_todas_previsoes_fec["cod_unidade"].nunique()}')
    print()
    print(df_todas_previsoes_fec.head(10))
else:
    print('⚠ Nenhuma previsão FEC gerada para exportar.')

✓ CSV final salvo em: /content/drive/MyDrive/API-6/previsoes_fec_prophet_todas_unidades.csv
  Total de linhas     : 30768
  Unidades processadas: 2564

   cod_unidade indicador       mae        data  previsao  previsao_lower  \
0          964       FEC  0.200297  2026-03-01  0.333250       -0.012570   
1          964       FEC  0.200297  2026-04-01  0.045582       -0.292654   
2          964       FEC  0.200297  2026-05-01  0.112273       -0.214639   
3          964       FEC  0.200297  2026-06-01 -0.040236       -0.400249   
4          964       FEC  0.200297  2026-07-01  0.046968       -0.281905   
5          964       FEC  0.200297  2026-08-01  0.009624       -0.332894   
6          964       FEC  0.200297  2026-09-01  0.095569       -0.244508   
7          964       FEC  0.200297  2026-10-01  0.230683       -0.088792   
8          964       FEC  0.200297  2026-11-01  0.150394       -0.195051   
9          964       FEC  0.200297  2026-12-01  0.411753        0.066656   

   previsao